# Jour 2 · Premier réseau Dense sur des images

## Objectifs

- comprendre une couche Dense et une sortie softmax
- entraîner et valider un petit classifieur d’images
- analyser les erreurs au-delà de l’accuracy


## Plusieurs neurones forment une couche

Une couche **Dense**, ou entièrement connectée, contient plusieurs neurones. Chaque neurone reçoit toutes les sorties de la couche précédente. Une succession de couches transforme progressivement le vecteur d'entrée en scores de classes.


![Réseau Dense reliant plusieurs couches](../ressources/illustrations/jour_02/02_reseau_dense.png)

*Chaque trait représente un poids distinct appris pendant l’entraînement.*


## Aplatir une image

Notre image 8 × 8 contient 64 pixels. Une couche `Flatten` range ces pixels dans un vecteur de 64 nombres. Le réseau Dense peut alors les combiner, mais il perd l'organisation spatiale explicite : deux pixels voisins dans l'image ne bénéficient pas d'un traitement spécial.

La dernière couche possède dix neurones, un par chiffre. **Softmax** transforme leurs scores en dix valeurs positives dont la somme vaut environ 1. Nous les lirons comme des confiances relatives entre les classes.


![Architecture Flatten Dense Dropout Softmax](../ressources/illustrations/jour_02/02_flatten_softmax.png)

La **couche d’entrée** décrit seulement la forme attendue. `Flatten` ne possède aucun poids : elle réorganise 8 × 8 valeurs en une liste de 64 valeurs.

`Dense(32)` apprend une représentation intermédiaire. Chacun de ses 32 neurones combine les 64 entrées. La couche finale possède dix neurones, un par classe.

Avant Softmax, le réseau produit des scores bruts parfois appelés **logits**. Softmax transforme ces scores en valeurs positives dont la somme vaut 1. `argmax` renvoie l’indice de la valeur la plus grande : cet indice devient la classe prédite.


In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf


tf.keras.utils.set_random_seed(42)
try:
    tf.config.threading.set_intra_op_parallelism_threads(1)
    tf.config.threading.set_inter_op_parallelism_threads(1)
except RuntimeError:
    pass
plt.style.use("seaborn-v0_8-whitegrid")
print("TensorFlow", tf.__version__)

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

digits = load_digits()
images = digits.images.astype("float32")
labels = digits.target.astype("int64")
X_train, X_test, y_train, y_test = train_test_split(
    images, labels, test_size=0.25, stratify=labels, random_state=42
)
X_train = X_train / 16.0
X_test = X_test / 16.0
print("Train :", X_train.shape, "Test :", X_test.shape)


## Aider le réseau à généraliser

Deux outils simples limitent le surapprentissage :

- **Dropout** désactive aléatoirement une partie des neurones pendant l'entraînement. Le réseau dépend moins d'un chemin unique. En prédiction, tous les neurones sont utilisés.
- **EarlyStopping**, ou arrêt anticipé, surveille la loss de validation. Si elle ne s'améliore plus pendant plusieurs époques, l'entraînement s'arrête et les meilleurs poids sont restaurés.

Ces techniques n'améliorent pas automatiquement tous les modèles. Elles donnent surtout un cadre pour éviter de continuer à mémoriser les données d'entraînement.


![Différence du Dropout entre entraînement et inférence](../ressources/illustrations/jour_02/02_dropout_train_inference.png)

`Dropout(0.20)` masque environ 20 % des sorties de la couche précédente à chaque passage d’entraînement. Le sous-ensemble masqué change aléatoirement, ce qui pousse le réseau à ne pas dépendre d’un seul chemin.

Pendant `evaluate` et `predict`, Dropout est désactivé. Toutes les unités participent, avec le comportement adapté automatiquement par Keras.

`EarlyStopping` joue un autre rôle : il observe une métrique de validation au fil des époques. `patience=4` tolère quatre époques sans amélioration avant l’arrêt. `restore_best_weights=True` remet ensuite les poids correspondant au meilleur résultat observé, pas ceux de la dernière époque.


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(8, 8)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.20),
    tf.keras.layers.Dense(10, activation="softmax"),
])
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

arret_anticipe = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True,
)

history = model.fit(
    X_train, y_train,
    validation_split=0.20,
    epochs=40,
    batch_size=32,
    callbacks=[arret_anticipe],
    verbose=0,
)
print("Époques réellement effectuées :", len(history.history["loss"]))
model.summary()


## Pourquoi cette loss ?

`sparse_categorical_crossentropy` compare la classe entière attendue, par exemple `7`, avec les dix valeurs produites par Softmax. Le mot *sparse* indique que les labels sont stockés comme des entiers et non comme dix colonnes contenant des zéros et un seul `1`.

Cette loss pénalise fortement le réseau lorsqu’il attribue peu de score à la bonne classe. L’accuracy, elle, regarde uniquement si la classe arrivée première est correcte. La loss fournit donc un signal plus fin pour apprendre, tandis que l’accuracy est souvent plus simple à communiquer.

`validation_split=0.20` réserve 20 % de `X_train` pour calculer `val_loss` et `val_accuracy`. Ces lignes ne servent pas à corriger les poids.

### Lire les réglages du modèle

- `Dense(32)` définit la largeur de la représentation intermédiaire ;
- `batch_size=32` traite 32 images avant chaque correction ;
- `epochs=40` fixe un maximum de 40 passages sur le train ;
- `callbacks=[arret_anticipe]` autorise un arrêt plus tôt ;
- `verbose=0` masque le journal détaillé sans modifier l’entraînement.

`model.summary()` affiche la forme de sortie de chaque couche et son nombre de paramètres. Un paramètre est une valeur apprise. Plus un réseau en possède, plus sa capacité et son coût de calcul peuvent augmenter.


In [ ]:
history_df = pd.DataFrame(history.history)
history_df[["loss", "val_loss"]].plot(figsize=(8, 3), title="Loss du réseau Dense")
plt.xlabel("époque")
plt.show()

test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print("Accuracy test :", round(test_accuracy, 3))


## Lire les courbes train et validation

`history.history` mémorise les valeurs mesurées après chaque époque. `loss` concerne les exemples utilisés pour apprendre ; `val_loss` concerne les exemples de validation mis de côté.

Si les deux diminuent, le modèle progresse généralement. Si la loss du train continue de baisser tandis que `val_loss` remonte durablement, le réseau commence probablement à mémoriser des particularités du train.

Le test intervient ensuite avec `model.evaluate`. Son accuracy estime la proportion d’images correctement classées sur des exemples restés hors de l’apprentissage et de l’arrêt anticipé.


![Courbes de loss avec et sans surapprentissage](../ressources/illustrations/jour_02/02_surapprentissage.png)

*Lorsque la validation remonte pendant que le train continue de baisser, le réseau mémorise davantage qu’il ne généralise.*


In [ ]:
probabilities = model.predict(X_test, verbose=0)
predictions = probabilities.argmax(axis=1)
cm = confusion_matrix(y_test, predictions)
pd.DataFrame(cm, index=range(10), columns=range(10))


### De la matrice de scores aux classes

`model.predict` renvoie une ligne de dix scores par image. `argmax(axis=1)` cherche la position du plus grand score sur chaque ligne. La matrice de confusion compare alors ces classes prédites aux labels réels.

Un score Softmax élevé décrit la préférence du modèle entre les dix choix proposés. Il ne garantit pas que l’image appartient réellement à cette classe, notamment si elle est très différente des images d’entraînement.


## Regarder les erreurs concrètes

Une matrice de confusion montre quelles classes sont confondues. Pour une application d'images réelle, il faut ensuite afficher les exemples mal classés : image floue, angle inhabituel, objet partiellement masqué ou label incorrect.


## À vous de jouer — afficher des erreurs

Affichez jusqu’à cinq images mal classées avec le label réel et la prédiction dans le titre.


In [ ]:
# Utilisez np.flatnonzero(predictions != y_test), puis plt.subplots.
pass
